# Regional Inequality from Outer Space

**Predicting GDP from nighttime lights and building inequality indices in Python** — a beginner-friendly replication of Lessmann & Seidel (2017): turn satellite nighttime lights into predicted regional GDP, build five population-weighted inequality indices from scratch, explore the cross-country dynamics of regional inequality, and estimate the regional Kuznets curve, its determinants, and a Conley spatial-HAC robustness check with PyFixest.

## Colab setup

Run this first on Google Colab to install the econometrics stack. Colab ships Python ≥ 3.10, so `pyfixest` imports cleanly (it requires Python 3.10+). `pandas`, `numpy`, and `matplotlib` are already present on Colab.

In [ ]:
# Colab Python is >= 3.10, so pyfixest imports fine.
!pip install -q pyfixest linearmodels statsmodels maketables

## 1. Overview

A government can tell you its country's GDP, but rarely the GDP of each province inside it. Two countries with identical national income can look completely different on the inside — one with a single booming capital surrounded by poor hinterlands, the other with broadly shared prosperity. To study that *internal* geography of income at a global scale, Lessmann and Seidel (2017) had a simple but powerful idea: **let satellites do the accounting**. Brighter places at night are, on average, richer places, so nighttime light can stand in for income where official statistics do not exist.

This notebook rebuilds their pipeline in Python, end to end. We start from light and a handful of controls, predict regional income, turn many regional incomes into a single inequality number per country, and finally ask the classic question: does regional inequality first rise and then fall as countries develop — the spatial version of the **Kuznets curve**?

The pipeline has four stages: light becomes income, income becomes inequality, and inequality becomes the object of study (the Kuznets curve and its drivers).

In this notebook you will:

- **Predict** regional GDP per capita from nighttime lights and controls, and form the predictions explicitly.
- **Construct** five population-weighted inequality indices from first principles, and see exactly how population weights change the answer.
- **Explore** the cross-country dynamics of regional inequality across time and world regions.
- **Estimate** the regional Kuznets curve, its determinants, and a spatially-robust standard error using PyFixest.
- **Distinguish** a prediction model from a causal claim, and a fixed-effects estimate from a random-effects one.

## 2. Key concepts at a glance

- **Nighttime lights as an income proxy.** The brightness a satellite records over a place at night, used as a stand-in for that place's economic output. Lights correlate with income because electricity use, roads, and activity all glow. They are imperfect — deserts and oil flares mislead — which is why we *predict* income from light rather than equate the two. (Turning brightness into a predicted income more than doubles its usefulness for measuring inequality: Gini correlation 0.49 vs 0.21.)
- **Light-to-GDP elasticity $\beta_1$.** The percent change in predicted regional GDP per capita for a 1% change in light per pixel, holding controls fixed. In the preferred specification $\beta_1 = 0.102$: a 10% brighter region is predicted to be about 1% richer, once national income and geography are controlled for.
- **Population-weighted inequality index.** A summary of how unequally income is spread across a country's regions, where each region counts in proportion to how many people live there. Germany 2010, built from its 16 regions, has a population-weighted Gini of 0.028 — low, because German regions are close in income.
- **The role of population weights.** Whether each region counts once (equal weight) or by its population changes the inequality number. Across country-years the weighted and unweighted Gini correlate 0.75; weighting lowers the average Gini by about 0.003.
- **The spatial Kuznets curve.** The hypothesis that regional inequality rises during early development, then falls as countries converge internally — an inverted U (or, with a third act at high income, an N) in inequality against log GDP per capita. The cubic in log income has coefficients $0.293 / -0.032 / 0.001$.
- **Conley (spatial-HAC) standard errors.** Standard errors that allow nearby regions' errors to be correlated, because a shock to one region usually spills into its neighbours. The light elasticity's SE rises from 0.013 (independent) to 0.026–0.037 (Conley, 1,000–5,000 km), but the estimate of 0.190 still sits far from zero.

## 3. Setup and imports

We use **pandas** and **numpy** for data work, **matplotlib** for figures, [**PyFixest**](https://py-econometrics.github.io/pyfixest/) for the panel fixed-effects regressions (its `feols` mirrors the R package `fixest`), **linearmodels** for the one random-effects table PyFixest cannot estimate, and **statsmodels** for a convenience regression behind one figure. PyFixest needs Python 3.10 or newer.

The site palette keeps the figures consistent: steel blue for primary data, warm orange for fitted lines and reference lines, near-black for the curves we want to stand out.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyfixest as pf
import maketables as mt
from linearmodels.panel import RandomEffects
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")
np.random.seed(42)

# --- Site colour palette (light theme) ---
STEEL, ORANGE, INK, TEAL = "#6a9bcc", "#d97757", "#141413", "#00d4c8"
GREY = "#b0a89a"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.bbox": "tight",
    "font.size": 11, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.edgecolor": INK, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": INK, "ytick.color": INK,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.grid": True, "grid.color": "#e6e3dd", "grid.linewidth": 0.7,
})

We load the bundled CSVs straight from GitHub so the notebook runs unchanged in Google Colab, falling back to a local `data/` folder when you run it offline.

In [ ]:
BASE = ("https://raw.githubusercontent.com/cmg777/starter-academic-v501/"
        "master/content/post/python_kuznets_dmsp/data/")


def load(name):
    """Read a bundled CSV from GitHub, falling back to a local data/ copy."""
    try:
        return pd.read_csv(BASE + name)
    except Exception:
        return pd.read_csv("data/" + name)

## 4. The data: sources and construction

This section documents the data behind every number in the post: what each file is for, where each variable came from, how it was built, and what it looks like descriptively. The exhaustive column-by-column reference is in Appendix A.

### 4.1 Three views of the world

The replication ships three "views" of the same world. The **region-year** files (`Prediction_Data.csv`, `Table_2_data.csv`, `Table_B4_data.csv`) describe individual subnational regions: their lights, their observed and predicted income, their populations and coordinates. The **country-year** files (`Table_3_data.csv`, `Table_4_data.csv`, `Figure_5_data.csv`) describe whole countries, each already carrying the inequality indices computed from its regions. We read all six.

In [ ]:
pred = load("Prediction_Data.csv")   # region-year: lights -> GDP training set
t2   = load("Table_2_data.csv")      # region-year: inequality-index inputs
t3   = load("Table_3_data.csv")      # country-year: Kuznets data
t4   = load("Table_4_data.csv")      # country-year: determinants
tb4  = load("Table_B4_data.csv")     # region-year: lat/lon for spatial errors
f5   = load("Figure_5_data.csv")     # country-year: regional vs personal Gini

for name, df in [("Prediction_Data", pred), ("Table_2_data", t2),
                 ("Table_3_data", t3), ("Table_4_data", t4),
                 ("Table_B4_data", tb4), ("Figure_5_data", f5)]:
    print(f"{name:16s} {df.shape[0]:5d} rows x {df.shape[1]:2d} cols")

# Map each country to its World Bank region group (from the region dummies).
GRP_COLS = ["eap", "eca", "lac", "mena", "sa", "ssa"]
GRP_NAME = {"eap": "East Asia & Pacific", "eca": "Europe & Central Asia",
            "lac": "Latin America & Carib.", "mena": "Mid. East & N. Africa",
            "sa": "South Asia", "ssa": "Sub-Saharan Africa"}


def row_group(r):
    for g in GRP_COLS:
        if r.get(g, 0) == 1:
            return GRP_NAME[g]
    return "N. America & high-inc."


pred["wb_group"] = pred.apply(row_group, axis=1)
pred["satyear"]  = sum(i * pred[f"satyear_{i}"] for i in range(1, 8)).astype(int)
pred["group_id"] = pred["wb_group"]
country_group = pred.groupby("Country_ISO")["wb_group"].agg(
    lambda s: s.value_counts().index[0])
n_reg = pred["code_Coutry_Region"].nunique()
n_cty = pred["Country_ISO"].nunique()
print(f"\nTraining sample: {pred.shape[0]} region-years, "
      f"{n_reg} regions, {n_cty} countries.")

The region-year files each hold 5,258 rows — these are the 1,504 regions, in 81 countries, that have *both* an observed GDP figure and a light reading, the sample used to calibrate the lights model. The country-year files hold 3,675 rows spanning 180 countries and the years 1992–2012. We calibrate and predict at the region level, then measure inequality and run the Kuznets regressions at the country level.

### 4.2 The six files at a glance

Six CSVs, each a tidy panel keyed by country (and, for the region files, by region) and year.
The complete column inventory for every file is in [Appendix A.1](#a1-the-six-datasets-in-detail);
here is what each file is *for* and what it carries.

- **`Prediction_Data.csv`** — *region-year* (5,258 × 30; 1,504 regions in 81 countries; 1992–2010).
  **Purpose:** the training sample that calibrates the light→income model (Table 1). These are the
  regions that have *both* an observed GDP figure (Gennaioli et al. 2014) and a light reading.
  **Components:** identifiers (`Country_ISO`, `code_Coutry_Region`, `id_t_j` = year+ISO); observed
  income (`GDP_pc_Region`, `log_GDP_pc_Region`); the model regressors (`log_Light_ppix_Region`,
  `log_GDP_pc_Country`, log top-/low-coded pixel counts, `log_area`, `log_region`, their
  interaction); World-Bank region-group dummies (`eap`…`ssa`); satellite-configuration dummies
  (`satyear_1`–`satyear_7`).
- **`Table_2_data.csv`** — *region-year* (5,258 × 8; same training frame). **Purpose:** inputs to
  *validate* the inequality indices — it pairs predicted and observed regional income with
  region/country light and population. **Components:** `pred_GDP_pc_Region`, `GDP_pc_Region`,
  `Light_Region`, `Light_Country`, `Pop_Region`, `Pop_Country`.
- **`Table_3_data.csv`** — *country-year* (3,675 × 9; 180 countries; 1992–2012). **Purpose:** the
  Kuznets dataset — national income plus the five population-weighted inequality indices built from
  predicted regional income. **Components:** `GDP_pc_Country` and `GINIW_`, `COVW_`, `GE_1W_`,
  `GE_0W_`, `GE_m1W_pred_GDP_pc`.
- **`Table_4_data.csv`** — *country-year* (3,675 × 17; 180 countries; 1992–2012). **Purpose:** the
  determinants dataset — the Kuznets variables plus the structural correlates of regional
  inequality. **Components:** `GINIW_pred_GDP_pc`, `GDP_pc_Country`, `Pop_Country`, and the
  determinants `Resources_rents_share_of_GDP`, `Arable_land`, `Trade_GDP_share`, `FDI_share_of_GDP`,
  `area`, `price_gasoline`, `Aid`, `School_enrollment_secondary`, `GINIW_Eth_light`, `Polity2`,
  `fedelupd2`.
- **`Table_B4_data.csv`** — *region-year* (5,258 × 14; training frame). **Purpose:** the
  spatial-robustness dataset — it adds each region's centroid so the Conley spatial-HAC standard
  errors (§11) can down-weight distant regions. **Components:** `Latitude`, `Longitude`,
  `log_GDP_pc_Region`, `log_Light_ppix_Region`, `satyear_1`–`satyear_7`.
- **`Figure_5_data.csv`** — *country-year* (3,675 × 5; 180 countries; 1992–2012). **Purpose:** the
  regional-versus-personal comparison (§12) — it sets the regional Gini beside a national
  interpersonal income Gini. **Components:** `GINIW_pred_GDP_pc` and `Giniall` (the personal Gini,
  observed for only 153 countries / 1,330 country-years).

### 4.3 How the key variables were built

Every variable above is the end of a construction chain that begins with raw satellite imagery
and public databases; tracing that chain is what makes the numbers interpretable.

- **Nighttime lights.** The light data are the DMSP-OLS *stable lights* product processed by the
  U.S. NOAA/National Geophysical Data Center: a digital number from 0 (dark) to 63 (saturated) for
  every ≈0.86 km² pixel, available annually from 1992. The authors average the light per pixel
  within each region and, following Hodler and Raschky (2014), add 0.01 where a region would
  otherwise read zero so the log is defined. Two censoring problems matter — bright cities
  top-code at 63, sparse areas bottom-code at 0 — which is why the prediction model also carries
  the counts of top- and low-coded pixels.
- **Sub-national boundaries.** Regions are the 1st-level administrative units (states, provinces,
  cantons) from the GADM database — roughly OECD TL2 / EUROSTAT NUTS1 — 3,166 regions across 180
  countries. The gridded light and population rasters are aggregated to these polygons.
- **Observed regional income.** The observed regional GDP per capita used to *train* the model
  comes from Gennaioli et al. (2014): GDP per capita in constant 2005 PPP US\$ for 1,503 regions
  in 82 countries, an unbalanced panel built from OECD, national-statistics, and
  human-development-report sources.
- **Population.** Regional population comes from the Gridded Population of the World (GPW) v3 raster
  (CIESIN): population density times region area, rounded up so the minimum is one, with the
  5-year survey waves interpolated to annual values.
- **Predicted regional income.** Because observed regional income exists for only ~80 countries,
  the model in §6 regresses log observed regional income on log light per pixel plus controls
  (country income, top-/low-coded pixel counts, number of regions, area and their interaction, and
  World-Bank region-group and satellite fixed effects) on the training sample, then *predicts*
  regional income for all 3,166 regions in 180 countries (1992–2012). The calibrated light
  elasticity is 0.102. Country-level controls come from the World Bank's World Development
  Indicators (WDI) and the CIA World Factbook.
- **Inequality indices.** From the predicted regional incomes, §7 builds five population-weighted
  indices per country-year — the Gini (`GINIW`), the coefficient of variation (`COVW`), and the
  generalized-entropy family GE(−1), GE(0) = mean log deviation, GE(1) = Theil — each weighting a
  region by its share of the national population so sparsely-populated outliers (e.g. Canada's
  Northern Territories) do not dominate.
- **Determinants.** The structural correlates in §10 are mostly WDI series — resource rents,
  arable-land share, trade and FDI shares, the gasoline pump price, net aid, and secondary-school
  enrolment — plus the Polity IV democracy score (Center for Systemic Peace, rescaled to
  \[−1, +1]), a federalism dummy, and an *ethnic-inequality* index that applies the same
  population-weighted light-Gini to ethnic homelands (GREG geo-referencing, Weidmann et al. 2010;
  method of Alesina et al. 2016).

### 4.4 Descriptive statistics

Two summary tables give the variables' shape, split by unit of observation (region files 1992–2010,
country files 1992–2012). Because the data are panels, each row also shows the variable's **mean in
the first and last year**. The post renders the full tables (every substantive variable) as figures;
here we print a representative subset.

In [ ]:
def _f(v):
    if pd.isna(v):
        return "--"
    a = abs(v)
    return f"{v:,.0f}" if a >= 1000 else (f"{v:.2f}" if a >= 1 else f"{v:.4f}")


def summarise_panel(spec, y0, y1):
    stat_fns = [("mean", "mean"), ("median", "median"), ("sd", "std"),
                ("min", "min"), ("max", "max")]
    rows = {}
    for label, df, col, scale in spec:
        s = pd.to_numeric(df[col], errors="coerce") * scale
        yr = df["year"]; obs = sorted(yr[s.notna()].unique())
        def ys(t, last=False):
            v = s[yr == t]
            return v if (v.notna().any() or not obs) else s[yr == (obs[-1] if last else obs[0])]
        vi, vf = ys(y0), ys(y1, last=True)
        rows[label] = [_f(getattr(vi, fn)()) if k == 0 else _f(getattr(vf, fn)())
                       for _, fn in stat_fns for k in (0, 1)]
    cols = pd.MultiIndex.from_tuples([(lab, str(y)) for lab, _ in stat_fns for y in (y0, y1)])
    return pd.DataFrame.from_dict(rows, orient="index", columns=cols)


region_spec = [("Observed GDP p.c. (region, $)", pred, "GDP_pc_Region", 1),
               ("Predicted GDP p.c. (region, $)", t2, "pred_GDP_pc_Region", 1),
               ("Log light per pixel (region)", pred, "log_Light_ppix_Region", 1),
               ("Population (region)", pred, "Pop_Region", 1)]
country_spec = [("GDP p.c. (country, $)", t3, "GDP_pc_Country", 1),
                ("Regional Gini (GINIW)", t3, "GINIW_pred_GDP_pc", 1),
                ("Resource rents (% GDP)", t4, "Resources_rents_share_of_GDP", 1),
                ("Gasoline price ($/L)", t4, "price_gasoline", 1),
                ("Personal income Gini", f5, "Giniall", 1)]
display(mt.MTable(summarise_panel(region_spec, 1992, 2010)).make("gt"))
mt.MTable(summarise_panel(country_spec, 1992, 2012)).make("gt")

### 4.5 Exploratory data analysis

Summary tables hide how the *whole distribution* moves over time. A box-plot over time restores that:
we bin the years into the five 5-year periods used later (§8) and, per period, draw a box of the
variable's distribution across units (each unit contributes its period mean). Reading left-to-right
shows the time dynamics; the box height shows the cross-sectional spread. We do it for the
region-level variables and then the country-level ones, so you get a feel for every dataset.

In [ ]:
def period_boxes(ax, df, unit, col, title, logy=False):
    d = df[df.year.between(1990, 2014)].assign(
        p=lambda x: pd.cut(x.year, [1989, 1994, 1999, 2004, 2009, 2014],
                           labels=["90-94", "95-99", "00-04", "05-09", "10-14"]))
    g = (d[["p", col]] if unit is None
         else d.groupby([unit, "p"], observed=True)[col].mean().reset_index())
    cats = [c for c in ["90-94", "95-99", "00-04", "05-09", "10-14"] if (g["p"] == c).any()]
    bp = ax.boxplot([pd.to_numeric(g.loc[g["p"] == c, col], errors="coerce").dropna() for c in cats],
                    patch_artist=True, showfliers=False, widths=0.6)
    for b in bp["boxes"]:
        b.set(facecolor=STEEL, alpha=0.65, edgecolor=INK)
    for m in bp["medians"]:
        m.set(color=ORANGE, linewidth=2)
    ax.set_xticks(range(1, len(cats) + 1)); ax.set_xticklabels(cats)
    if logy:
        ax.set_yscale("log")
    ax.set_title(title, fontsize=10); ax.set_xlabel("5-year period")

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
period_boxes(axes[0, 0], pred, "code_Coutry_Region", "log_Light_ppix_Region", "Log light per pixel")
period_boxes(axes[0, 1], pred, "code_Coutry_Region", "GDP_pc_Region", "Observed GDP p.c. ($, log)", logy=True)
period_boxes(axes[1, 0], t2, None, "pred_GDP_pc_Region", "Predicted GDP p.c. ($, log)", logy=True)
period_boxes(axes[1, 1], pred, "code_Coutry_Region", "Pop_Region", "Population (log)", logy=True)
fig.suptitle("Region-level variables over time (training sample)", fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.95)); plt.show()

cc = [(t3, "GDP_pc_Country", "GDP p.c. ($, log)", True),
      (t3, "GINIW_pred_GDP_pc", "Regional Gini", False),
      (t4, "Resources_rents_share_of_GDP", "Resource rents (% GDP)", False),
      (t4, "Trade_GDP_share", "Trade / GDP", False),
      (t4, "price_gasoline", "Gasoline price ($/L)", False),
      (f5, "Giniall", "Personal income Gini", False)]
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, (df, col, title, logy) in zip(axes.flat, cc):
    period_boxes(ax, df, "Country_ISO", col, title, logy=logy)
fig.suptitle("Country-level variables over time (180 countries)", fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.95)); plt.show()

## 5. Cross-country dynamics of inequality

Before predicting or regressing anything, it pays to *see* the data. This section maps the landscape: how the key variables are distributed, how regional inequality has moved over two decades, how it differs across world regions, and how the five inequality indices relate to one another.

In [ ]:
IDX = ["GINIW_pred_GDP_pc", "COVW_pred_GDP_pc", "GE_1W_pred_GDP_pc",
       "GE_0W_pred_GDP_pc", "GE_m1W_pred_GDP_pc"]
IDX_LAB = {"GINIW_pred_GDP_pc": "Gini", "COVW_pred_GDP_pc": "CV",
           "GE_1W_pred_GDP_pc": "Theil GE(1)", "GE_0W_pred_GDP_pc": "MLD GE(0)",
           "GE_m1W_pred_GDP_pc": "GE(-1)"}
eda = t3.copy()
eda["wb_group"] = eda["Country_ISO"].map(country_group)
eda["log_GDPpc"] = np.log(eda["GDP_pc_Country"])

### 5.1 Distributions of the key variables

We begin with three histograms: the log of nighttime light per pixel and the log of regional GDP per capita (both at the region level), and the population-weighted regional Gini (at the country level). Looking at distributions first tells us whether variables are skewed, bounded, or multi-modal.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
axes[0].hist(pred["log_Light_ppix_Region"].dropna(), bins=40, color=STEEL,
             edgecolor="white")
axes[0].set(title="Log nighttime light per pixel\n(region-year)",
            xlabel="log light per pixel", ylabel="count")
axes[1].hist(np.log(pred["GDP_pc_Region"].dropna()), bins=40, color=ORANGE,
             edgecolor="white")
axes[1].set(title="Log regional GDP per capita\n(region-year)",
            xlabel="log GDP per capita", ylabel="count")
axes[2].hist(eda["GINIW_pred_GDP_pc"].dropna(), bins=40, color=TEAL,
             edgecolor="white")
axes[2].set(title="Regional inequality GINIW\n(country-year)",
            xlabel="population-weighted Gini", ylabel="count")
fig.tight_layout()
plt.show()

print("GINIW: mean={:.3f}, median={:.3f}, max={:.3f}".format(
    eda["GINIW_pred_GDP_pc"].mean(), eda["GINIW_pred_GDP_pc"].median(),
    eda["GINIW_pred_GDP_pc"].max()))

Log light and log income are both roughly bell-shaped — taking logs tames their heavy right skew, which is why the calibration model works in logs. The regional Gini is right-skewed and bounded below by zero, with a mean of 0.064 and a maximum of 0.163: most countries are internally fairly equal, but a long tail of countries has very uneven regions.

### 5.2 Inequality and income over time

Has regional inequality risen or fallen as the world grew richer? We average the regional Gini and log GDP per capita across all countries in each year from 1992 to 2012 and plot them on a shared timeline.

In [ ]:
yr = (eda[(eda.year >= 1992) & (eda.year <= 2012)]
      .groupby("year").agg(GINIW=("GINIW_pred_GDP_pc", "mean"),
                           logGDP=("log_GDPpc", "mean")).reset_index())
fig, ax1 = plt.subplots(figsize=(7, 4.2))
ax1.plot(yr.year, yr.GINIW, color=STEEL, marker="o", lw=2, label="mean GINIW")
ax1.set(xlabel="year", ylabel="mean regional inequality (GINIW)")
ax1.tick_params(axis="y", labelcolor=STEEL)
ax2 = ax1.twinx()
ax2.plot(yr.year, yr.logGDP, color=ORANGE, marker="s", lw=2,
         label="mean log GDP p.c.")
ax2.set_ylabel("mean log GDP per capita", color=ORANGE)
ax2.tick_params(axis="y", labelcolor=ORANGE)
ax2.grid(False)
ax1.set_title("Average regional inequality and income, 1992-2012")
fig.tight_layout()
plt.show()

print(f"GINIW {yr.GINIW.iloc[0]:.4f} ({int(yr.year.iloc[0])}) -> "
      f"{yr.GINIW.iloc[-1]:.4f} ({int(yr.year.iloc[-1])})")

As average world income climbed (orange, rising), average regional inequality fell from 0.070 in 1992 to 0.061 in 2012 (steel, declining). Globally, growth and *falling* within-country inequality went together over this period — a first hint that, on the downward arm of the Kuznets curve, development narrows regional gaps.

### 5.3 Inequality across world regions

We group countries into the World Bank's regions and draw a box plot of the regional Gini for each. A box plot shows the median (the orange line), the middle half of countries (the box), and the spread (the whiskers).

In [ ]:
order = (eda.groupby("wb_group")["GINIW_pred_GDP_pc"].median()
         .sort_values().index.tolist())
data_by = [eda.loc[eda.wb_group == g, "GINIW_pred_GDP_pc"].dropna()
           for g in order]
fig, ax = plt.subplots(figsize=(8.5, 4.6))
bp = ax.boxplot(data_by, vert=False, patch_artist=True, widths=0.6,
                showfliers=False)
for patch in bp["boxes"]:
    patch.set(facecolor=STEEL, alpha=0.65, edgecolor=INK)
for med in bp["medians"]:
    med.set(color=ORANGE, linewidth=2)
ax.set_yticklabels(order)
ax.set(xlabel="regional inequality (GINIW)",
       title="Regional inequality across World Bank regions")
fig.tight_layout()
plt.show()

grp_med = eda.groupby("wb_group")["GINIW_pred_GDP_pc"].median().sort_values()
print("median GINIW by region:")
for g, v in grp_med.items():
    print(f"  {g:26s} {v:.4f}")

Sub-Saharan Africa has the highest median regional inequality (0.096) — two and a half times that of North America and high-income countries (0.039) — with East Asia and Latin America close behind. Rich regions are not only richer on average; their *internal* income map is far more even. This cross-section already sketches the downward arm of a Kuznets relationship.

### 5.4 How the five indices co-move

The paper measures inequality five ways: the Gini, the coefficient of variation (CV), and three generalized-entropy indices — GE(−1), GE(0) (the mean log deviation), and GE(1) (the Theil index). Do they tell the same story? We compute their correlation matrix across all country-years.

In [ ]:
cmat = t3[IDX].corr()
fig, ax = plt.subplots(figsize=(5.6, 5))
im = ax.imshow(cmat.values, cmap="BuPu", vmin=0, vmax=1)
labs = [IDX_LAB[c] for c in IDX]
ax.set_xticks(range(5)); ax.set_xticklabels(labs, rotation=40, ha="right")
ax.set_yticks(range(5)); ax.set_yticklabels(labs)
for i in range(5):
    for j in range(5):
        ax.text(j, i, f"{cmat.values[i, j]:.2f}", ha="center", va="center",
                color="white" if cmat.values[i, j] > 0.6 else INK, fontsize=9)
ax.set_title("Co-movement of five inequality indices")
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.8, label="correlation")
fig.tight_layout()
plt.show()

print("corr(Gini, CV)    = %.3f" % cmat.loc["GINIW_pred_GDP_pc", "COVW_pred_GDP_pc"])
print("corr(Gini, Theil) = %.3f" % cmat.loc["GINIW_pred_GDP_pc", "GE_1W_pred_GDP_pc"])

All five indices correlate above 0.9 — the Gini and the CV move almost in lockstep (0.97). This is reassuring: whichever index we lead with, the qualitative findings will be the same. With the landscape mapped, we turn to the engine of the whole exercise — turning light into income.

## 6. Predicting GDP from nighttime lights

This is the first of the two construction stages, and the foundation of everything after it. The goal is a model that takes a region's nighttime brightness plus a few controls and returns a prediction of its income — a model we can then apply to regions that have *no* income statistics. We build it exactly as Table 1 of the paper does, calibrating on the 1,504 regions where income is observed.

### 6.1 The idea: light as a proxy for income

We regress the log of a region's GDP per capita on the log of its light per pixel, plus controls that absorb everything light should *not* be credited with — national income, geography, satellite generation, and broad world region:

$$y_r = \beta_0 + \beta_1 \ell_r + \beta_2 g_c + \gamma' X_r + \mu_g + \tau_s + \varepsilon_r$$

A region's log income $y_r$ is a baseline $\beta_0$, plus an elasticity $\beta_1$ times its log light $\ell_r$, plus a near-one-for-one adjustment $\beta_2$ for its country's log income $g_c$, plus geography controls $X_r$, plus a world-region effect $\mu_g$ and a satellite-generation effect $\tau_s$. The coefficient we care about is $\beta_1$ — the light-to-GDP elasticity.

### 6.2 Seven specifications in PyFixest

The paper builds the model up in seven steps, each adding fixed effects or controls, so we can watch the elasticity stabilise. We run all seven as fixed-effects/OLS models in PyFixest. The fixed effects go after the `|`; standard errors are clustered by country (`CRV1`).

In [ ]:
sat_d = [f"satyear_{i}" for i in range(1, 8)]

GEO = ("log_N_pix_top_cod_1_ppix + log_N_pix_low_cod_1_ppix + log_area + "
       "log_region + log_region_X_log_area")
fe_specs = {
    1: "log_GDP_pc_Region ~ log_Light_ppix_Region",
    2: "log_GDP_pc_Region ~ log_Light_ppix_Region | code_Coutry_Region+satyear",
    3: "log_GDP_pc_Region ~ log_Light_ppix_Region | Country_ISO+satyear",
    4: "log_GDP_pc_Region ~ log_Light_ppix_Region + log_GDP_pc_Country | Country_ISO+satyear",
    5: "log_GDP_pc_Region ~ log_Light_ppix_Region | group_id+satyear",
    6: "log_GDP_pc_Region ~ log_Light_ppix_Region + log_GDP_pc_Country | group_id+satyear",
    7: f"log_GDP_pc_Region ~ log_Light_ppix_Region + log_GDP_pc_Country + {GEO} | group_id+satyear",
}
fe_models, fe_b = {}, {}
for k, fml in fe_specs.items():
    m = pf.feols(fml, data=pred, vcov={"CRV1": "Country_ISO"})
    fe_models[k] = m
    fe_b[k] = float(m.coef()["log_Light_ppix_Region"])
print("PyFixest FE/OLS light elasticity by col (1)-(7): "
      + " / ".join(f"{fe_b[k]:.3f}" for k in range(1, 8)))
b_natgdp_fe7 = float(fe_models[7].coef()["log_GDP_pc_Country"])

# Table 1 as a maketables regression table (the seven specifications side by side)
et1 = mt.ETable([fe_models[k] for k in range(1, 8)],
                head_order="d",
                labels={"log_GDP_pc_Region": "log regional GDP per capita",
                        "log_Light_ppix_Region": "log light per pixel"},
                felabels={"code_Coutry_Region": "Region FE", "Country_ISO": "Country FE",
                          "group_id": "WB-group FE", "satyear": "Satellite FE"},
                coef_fmt="b:.3f* (se:.3f)", show_fe=True)
et1.make("gt")


The pooled elasticity of 0.359 (column 1) falls to 0.190 once we absorb region fixed effects (column 2), and falls further as national income and geography are added. Column 2 is worth remembering: 0.190 is the *clean within-region* elasticity, the number Section 10 stress-tests for spatial correlation. Column 7's fixed-effects elasticity is the smallest — once national income and broad region are absorbed there is little cross-region variation left for light to explain. But the paper did not use fixed effects here — it used random effects.

### 6.3 The random-effects sidebar

PyFixest estimates only fixed-effects and OLS models. The paper's Table 1, however, uses a **random-effects** estimator — it treats each region's intercept as a random draw and uses *both* the differences between regions and the changes within them. To reproduce the published numbers we briefly switch to `linearmodels.RandomEffects`.

In [ ]:
# WHY random effects (not fixed effects): fixed effects compare each region only
# with itself over time, so they cannot predict for a region OUTSIDE the sample
# (no intercept for it). Random effects treat each region's intercept as a draw
# from a common distribution -> ONE coefficient vector usable for ANY region,
# which is exactly what the paper needs to map income for every region on Earth.
panel = pred.set_index(["code_Coutry_Region", "year"])   # (region, year) panel index
cluster_id = pd.Categorical(panel["Country_ISO"].values).codes
clusters = pd.DataFrame({"c": cluster_id}, index=panel.index)
cdum = pd.get_dummies(panel["Country_ISO"], prefix="cty",
                      drop_first=True).astype(float)
gdum = pd.get_dummies(panel["wb_group"], prefix="grp",
                      drop_first=True).astype(float)
satm = panel[sat_d].astype(float)


def re_design(extra_cols):
    # always prepend a constant (the shared baseline intercept)
    return pd.concat([pd.Series(1.0, index=panel.index, name="const")]
                     + extra_cols, axis=1)


def re_fit(extra_cols):
    y = panel["log_GDP_pc_Region"]
    X = re_design(extra_cols)
    return RandomEffects(y, X).fit(cov_type="clustered", clusters=clusters)


light = panel[["log_Light_ppix_Region"]]
ngdp  = panel[["log_GDP_pc_Country"]]
geo   = panel[["log_N_pix_top_cod_1_ppix", "log_N_pix_low_cod_1_ppix",
               "log_area", "log_region", "log_region_X_log_area"]]
# fit the ladder of specifications; re7 is the full column-7 model
re1 = re_fit([light])
re3 = re_fit([light, cdum, satm])
re4 = re_fit([light, ngdp, cdum, satm])
re5 = re_fit([light, gdum, satm])
re6 = re_fit([light, ngdp, gdum, satm])
re7 = re_fit([light, ngdp, geo, gdum, satm])
re_b = {1: float(re1.params["log_Light_ppix_Region"]),
        2: fe_b[2],
        3: float(re3.params["log_Light_ppix_Region"]),
        4: float(re4.params["log_Light_ppix_Region"]),
        5: float(re5.params["log_Light_ppix_Region"]),
        6: float(re6.params["log_Light_ppix_Region"]),
        7: float(re7.params["log_Light_ppix_Region"])}
b_natgdp_re7 = float(re7.params["log_GDP_pc_Country"])
print("linearmodels RE light elasticity by col (1)-(7): "
      + " / ".join(f"{re_b[k]:.3f}" for k in range(1, 8)))
print(f"RE col 7 light elasticity        = {re_b[7]:.3f}")
print(f"RE col 7 national-GDP elasticity = {b_natgdp_re7:.3f}")

The random-effects elasticity in column 7 is **0.102** — exactly the paper's number — versus the 0.049 we got with fixed effects. They differ because random effects keep the between-region information that the within estimator throws away. The national-GDP elasticity of **0.889** confirms regional income tracks national income almost one-for-one, with light supplying the residual subnational detail. The two estimators agree exactly in column 2, the one true fixed-effects column (0.190).

### 6.4 Forming the predictions

A model is only useful if we can *predict* with it. We reconstruct the fitted log income for every region from the column-7 coefficients — multiply each region's characteristics by the estimated $\beta$'s and add them up — then exponentiate to get a predicted GDP per capita in dollars. Comparing the predictions to the observed values is the honest test of whether the calibration generalises.

In [ ]:
X7 = re_design([light, ngdp, geo, gdum, satm])
beta7 = re7.params.reindex(X7.columns).values
fitted_log = X7.values @ beta7
obs_log = panel["log_GDP_pc_Region"].values
pred_pc = np.exp(fitted_log)        # predicted GDP per capita, in dollars
obs_pc  = np.exp(obs_log)
r_pred = np.corrcoef(fitted_log, obs_log)[0, 1]
print(f"corr(predicted, observed log GDP per capita) = {r_pred:.3f} "
      f"over {len(fitted_log)} region-years")

fig, ax = plt.subplots(figsize=(5.4, 5.2))
ax.scatter(fitted_log, obs_log, s=10, facecolors="none",
           edgecolors=STEEL, alpha=0.5)
lo, hi = 4, 13
ax.plot([lo, hi], [lo, hi], color=ORANGE, lw=2, label="45° line (perfect fit)")
ax.set(xlim=(lo, hi), ylim=(lo, hi),
       xlabel="predicted log GDP per capita (from lights)",
       ylabel="observed log GDP per capita",
       title=f"Predicted vs observed regional income\nPearson r = {r_pred:.3f}")
ax.legend(loc="upper left", frameon=False)
fig.tight_layout()
plt.show()

Predicted and observed log income correlate **0.925** across all 5,258 region-years, and the scatter hugs the 45° line across four orders of magnitude of income. The model is not memorising one income band; it generalises from the poorest regions to the richest. That is what licenses applying these coefficients to *every* region on Earth, including the tens of thousands with no income statistics, to build a complete global income map. With predicted income in hand, we can finally measure inequality.

## 7. Constructing the inequality indicators

This is the second construction stage. We now have a predicted income for every region; the task is to compress each country's many regional incomes into a single number that says how unequal they are — and to do it in a way that respects population.

### 7.1 From many regional incomes to one number

Every index starts from the same three ingredients. Let region $i$ have income $y_i$ and population $w_i$. The **population-weighted mean**, the **population shares**, and the **relative incomes** are

$$\bar y = \frac{\sum_i w_i y_i}{\sum_i w_i}, \qquad
  p_i = \frac{w_i}{\sum_j w_j}, \qquad
  r_i = \frac{y_i}{\bar y}.$$

In code, $y_i$ is `pred_GDP_pc_Region`, $w_i$ is `Pop_Region`. Weighting by population is the key design choice: a region matters in proportion to how many people experience its income.

### 7.2 The five indices from scratch

The **Gini** is the average absolute income gap between two randomly chosen people, scaled to lie in $[0, 1]$. The **generalized-entropy** family $GE(\alpha)$ varies in how sharply it reacts to gaps at the top or bottom of the distribution, and the **coefficient of variation** is the standard deviation over the mean:

$$G = \frac{\sum_i \sum_j w_i w_j \, |y_i - y_j|}{2 \left(\sum_i w_i\right)^2 \bar y},
  \qquad
  GE(0) = \sum_i p_i \ln\!\frac{1}{r_i}, \qquad
  GE(1) = \sum_i p_i \, r_i \ln r_i.$$

A crucial coding detail: the Gini uses the **absolute difference** $|y_i - y_j|$, summed over all pairs — not a product — which is the classic trap when writing a weighted Gini by hand.

In [ ]:
def ineq_indices(y, w):
    """Five population-weighted inequality indices from first principles.

    y = regional income, w = regional population. Returns Gini, GE(-1),
    GE(0)=MLD, GE(1)=Theil and the coefficient of variation (CV)."""
    # --- Step 1: clean the inputs (drop missing / non-positive) ------------
    y = np.asarray(y, float)
    w = np.asarray(w, float)
    ok = np.isfinite(y) & np.isfinite(w) & (w > 0) & (y > 0)
    y, w = y[ok], w[ok]
    if y.size < 2:
        return dict(GINIW=np.nan, GE_m1W=np.nan, GE_0W=np.nan,
                    GE_1W=np.nan, COVW=np.nan)
    # --- Step 2: the three ingredients (mean, shares, relative incomes) ----
    sw = w.sum()                             # total population
    mu = (w * y).sum() / sw                  # population-weighted mean income
    p = w / sw                               # population shares (sum to 1)
    r = y / mu                               # relative incomes (y_i / mean)
    # --- Step 3a: generalized-entropy family + coefficient of variation ----
    ge_m1 = 0.5 * ((p * r ** -1).sum() - 1)  # GE(-1): sensitive to the poorest
    ge_0 = (p * (-np.log(r))).sum()          # GE(0) = mean log deviation
    ge_1 = (p * r * np.log(r)).sum()         # GE(1) = Theil index
    ge_2 = 0.5 * ((p * r ** 2).sum() - 1)
    cv = np.sqrt(2 * ge_2)                   # coefficient of variation
    # --- Step 3b: Gini = population-weighted avg gap between people ---------
    # y[:,None]-y[None,:] is the matrix of all pairwise gaps; np.abs keeps the
    # SIZE of each gap (the classic trap is forgetting the absolute value);
    # np.outer(w, w) weights each pair by both populations.
    gini = (np.abs(y[:, None] - y[None, :]) * np.outer(w, w)).sum() \
        / (2 * sw ** 2 * mu)
    return dict(GINIW=gini, GE_m1W=ge_m1, GE_0W=ge_0, GE_1W=ge_1, COVW=cv)


def gini_unweighted(y):
    """Equal-weight Gini (every region counts once) for the weight comparison."""
    # note what is missing vs ineq_indices: no population weights, no np.outer
    y = np.asarray(y, float)
    y = y[np.isfinite(y) & (y > 0)]
    n = y.size
    if n < 2:
        return np.nan
    mu = y.mean()
    return np.abs(y[:, None] - y[None, :]).sum() / (2 * n ** 2 * mu)

This single function is the whole measurement apparatus. It takes a country-year's regional incomes and populations and returns all five indices. To trust them, we test the function on a country we can reason about.

In [ ]:
# Build the five indices per country-year on predicted income
rows = []
for (iso, yr_), g in t2.groupby(["Country_ISO", "year"]):
    rec = {"Country_ISO": iso, "year": yr_, "n_regions": len(g)}
    rec.update(ineq_indices(g["pred_GDP_pc_Region"], g["Pop_Region"]))
    rec["GINI_unw"] = gini_unweighted(g["pred_GDP_pc_Region"])
    rows.append(rec)
built = pd.DataFrame(rows)
print(f"built indices for {len(built)} country-years.")

### 7.3 A worked example: Germany

Germany is a good test case: 16 regions of broadly similar income, so we expect a *low* inequality number. We pull its 2010 regions and run them through the function by hand.

In [ ]:
deu = t2[(t2.Country_ISO == "DEU") & (t2.year == 2010)]
print("regions:", len(deu))
ex = ineq_indices(deu["pred_GDP_pc_Region"], deu["Pop_Region"])
print({k: round(v, 4) for k, v in ex.items()})

Germany's 16 regions yield a population-weighted Gini of **0.028** — very low, as expected for a country whose regions cluster near the national average. The Theil index (0.0016) and the others agree. A concrete, hand-checkable number like this is the sanity check that the formula is implemented correctly before we apply it to 180 countries.

### 7.4 The role of population weights

Does population weighting actually change anything? We recompute the Gini for every country-year *without* weights — letting every region count once — and compare.

In [ ]:
wcmp = built.dropna(subset=["GINIW", "GINI_unw"]).copy()
wcmp["diff"] = wcmp["GINIW"] - wcmp["GINI_unw"]
corr_wu = wcmp["GINIW"].corr(wcmp["GINI_unw"])
print(f"corr(weighted, unweighted)  = {corr_wu:.3f}")
print(f"mean(weighted - unweighted) = {wcmp['diff'].mean():+.4f}")

fig, ax = plt.subplots(figsize=(5.6, 5.4))
ax.scatter(wcmp["GINI_unw"], wcmp["GINIW"], s=10, facecolors="none",
           edgecolors=STEEL, alpha=0.45)
mx = max(wcmp["GINI_unw"].max(), wcmp["GINIW"].max()) * 1.02
ax.plot([0, mx], [0, mx], color=ORANGE, lw=2, label="equal-weight = weighted")
ax.set(xlim=(0, mx), ylim=(0, mx),
       xlabel="unweighted Gini (every region counts once)",
       ylabel="population-weighted Gini (GINIW)",
       title="The role of population weights")
ax.legend(loc="upper left", frameon=False)
fig.tight_layout()
plt.show()

The weighted and unweighted Gini correlate only **0.75** — far from identical — and weighting *lowers* inequality on average by 0.0034. Most points sit below the 45° line: population weighting pulls the index down because small, income-extreme regions (a tiny mining province, a remote capital) count for less when we weight by people. The lesson is general — **report your weighting**: the same country can look more or less unequal depending on whether you count regions or people, and "by people" is usually the policy-relevant choice.

### 7.5 Do our indices match the paper?

The from-scratch indices should reproduce the paper's Table 2 — the correlation between inequality measured from *predicted* income and inequality measured from *observed* income, contrasted with raw light density.

In [ ]:
t2c = t2.copy()
t2c["Light_pc_Region"] = t2c["Light_Region"] / t2c["Pop_Region"]
t2c = t2c[(t2c.year > 2000) & (t2c.year < 2013)].copy()
bases = {"pred": "pred_GDP_pc_Region", "obs": "GDP_pc_Region",
         "light": "Light_pc_Region"}
keys = ["GINIW", "GE_m1W", "GE_0W", "GE_1W", "COVW"]
recs = []
for (iso, yr_), g in t2c.groupby(["Country_ISO", "year"]):
    rec = {"Country_ISO": iso}
    for suf, var in bases.items():
        v = ineq_indices(g[var], g["Pop_Region"])
        for kk in keys:
            rec[f"{kk}_{suf}"] = v[kk]
    recs.append(rec)
percty = pd.DataFrame(recs).groupby("Country_ISO", as_index=False).mean()
lab5 = ["Gini", "GE(-1)", "MLD GE(0)", "Theil GE(1)", "CV"]
po = [percty[f"{k}_pred"].corr(percty[f"{k}_obs"]) for k in keys]
lo = [percty[f"{k}_light"].corr(percty[f"{k}_obs"]) for k in keys]
print(f"Table 2 correlations across {len(percty)} countries:")
print("  predicted vs observed:", [round(x, 2) for x in po])
print("  raw light vs observed:", [round(x, 2) for x in lo])

# Table 2 as a maketables correlation table
t2tab = pd.DataFrame({"Predicted income vs observed": [f"{v:.2f}" for v in po],
                      "Raw light vs observed": [f"{v:.2f}" for v in lo]}, index=lab5)
t2tab.index.name = "Inequality index"
mt.MTable(t2tab).make("gt")


Inequality computed from *predicted* income correlates with inequality from *observed* income at 0.49 for the Gini — more than double the 0.21 we get from raw light density, and the same pattern holds for all five indices. This is the payoff of the prediction step: turning light into income first, instead of treating brightness as income, roughly doubles how well we measure inequality. One honest caveat: our from-scratch indices are built on the ~1,500 regions that have *observed* income, whereas the paper's published series uses *every* subnational region on Earth — the two correlate 0.88, not 1.00, which is precisely why the paper had to predict income for all regions.

## 8. The regional Kuznets curve

Now the classic question. As countries grow richer, does regional inequality rise then fall? We regress the regional Gini on a cubic in log national income, with country and period fixed effects so the relationship is identified from each country's *own* changes over time, not from rich-vs-poor comparisons.

### 8.1 The cubic specification in PyFixest

We average the data into 5-year periods (to smooth annual noise), build the cubic terms, and estimate with country and period fixed effects clustered by country:

$$\text{GINIW}_{ct} = \beta_1 \ln Y_{ct} + \beta_2 (\ln Y_{ct})^2
   + \beta_3 (\ln Y_{ct})^3 + \alpha_c + \delta_t + u_{ct},$$

where $\alpha_c, \delta_t$ are country and period fixed effects. In code $\ln Y$ and its powers are `lg, lg2, lg3`, and the fixed effects are `Country_ISO + p5`.

In [ ]:
def collapse5(df, vars_):
    d = df[(df.year >= 1990) & (df.year <= 2014)].copy()
    d["p5"] = pd.cut(d.year, [1989, 1994, 1999, 2004, 2009, 2014],
                     labels=[1, 2, 3, 4, 5]).astype("int64")
    return d.groupby(["Country_ISO", "p5"], as_index=False)[vars_].mean()


agg3 = collapse5(t3, ["GDP_pc_Country"] + IDX)
agg3["lg"]  = np.log(agg3["GDP_pc_Country"])
agg3["lg2"] = agg3["lg"] ** 2
agg3["lg3"] = agg3["lg"] ** 3


def kuz_fit(dv, rhs):
    return pf.feols(f"{dv} ~ {rhs} | Country_ISO+p5", data=agg3,
                    vcov={"CRV1": "Country_ISO"})


k1 = kuz_fit("GINIW_pred_GDP_pc", "lg")
k2 = kuz_fit("GINIW_pred_GDP_pc", "lg + lg2")
k3 = kuz_fit("GINIW_pred_GDP_pc", "lg + lg2 + lg3")
k_other = {c: kuz_fit(c, "lg + lg2 + lg3") for c in IDX[1:]}
N3 = int(k3._N)
b3 = k3.coef()
print(f"GINIW cubic: {b3['lg']:.3f} / {b3['lg2']:.3f} / {b3['lg3']:.4f}")
print(f"N = {N3}  countries = {agg3.Country_ISO.nunique()}")

# Table 3 as a maketables regression table (7 columns)
et3 = mt.ETable([k1, k2, k3] + [k_other[c] for c in IDX[1:]],
                model_heads=["linear", "quadratic", "cubic", "", "", "", ""],
                labels={"GINIW_pred_GDP_pc": "Population-weighted regional Gini",
                        "COVW_pred_GDP_pc": "Coeff. of variation",
                        "GE_1W_pred_GDP_pc": "Theil index",
                        "GE_0W_pred_GDP_pc": "Mean log deviation",
                        "GE_m1W_pred_GDP_pc": "GE(−1)"},
                felabels={"Country_ISO": "Country FE", "p5": "Period FE"},
                coef_fmt="b:.3f* (se:.3f)", show_fe=True)
et3.make("gt")


The cubic coefficients are **0.293 / −0.032 / 0.001** — positive, negative, positive — exactly the paper's values. The positive linear term means inequality rises with income at low levels; the negative quadratic bends the curve down; the tiny positive cubic adds a faint upturn at the very top. This is an **N-shape**: a Kuznets hump with a third act. The same sign pattern holds for all four other indices, so the shape is not an artefact of the Gini.

### 8.2 Visualising the curve

Coefficients are abstract; a picture is not. We plot each country-period's regional Gini (net of period effects) against its log income, and overlay the fitted cubic.

In [ ]:
# Partial-residual plot: refit with EXPLICIT dummies (statsmodels) so we can read
# off the period effects, net them out of each point, then draw the cubic through
# the resulting cloud (the points then sit in a tight band around the curve).
# Step 1: refit the same cubic with explicit country + period dummies
mfe = smf.ols("GINIW_pred_GDP_pc ~ lg + lg2 + lg3 + C(Country_ISO) + C(p5)",
              agg3).fit()
rterms = ["lg", "lg2", "lg3"]
bb = {k: mfe.params[k] for k in rterms}
# Step 2: net each 5-year period's effect out of its points (period 1 = baseline 0)
peff = {1: 0.0}
for k in (2, 3, 4, 5):
    peff[k] = mfe.params.get(f"C(p5)[T.{k}]", 0.0)
agg3["partial"] = agg3["GINIW_pred_GDP_pc"] - agg3["p5"].map(peff)
# Step 3: recenter the curve to the average height of the cloud
cons = (agg3["partial"] - (bb["lg"] * agg3.lg + bb["lg2"] * agg3.lg2
        + bb["lg3"] * agg3.lg3)).mean()
# Step 4: evaluate the fitted cubic on a smooth grid of log incomes
xs = np.linspace(5.5, 11.8, 200)
ys = cons + bb["lg"] * xs + bb["lg2"] * xs ** 2 + bb["lg3"] * xs ** 3
fig, ax = plt.subplots(figsize=(6.4, 4.6))
ax.scatter(agg3.lg, agg3.partial, s=14, facecolors="none",
           edgecolors=STEEL, alpha=0.55)
ax.plot(xs, ys, color=INK, lw=2.4, label="fitted cubic")
ax.set(xlim=(5.5, 11.8), ylim=(0, 0.16), xlabel="log GDP per capita",
       ylabel="partial regional inequality (GINIW)",
       title="Regional inequality and development (Figure 4)")
ax.legend(loc="upper right", frameon=False)
fig.tight_layout()
plt.show()
print(f"Figure 4 cubic (OLS-dummies): {bb['lg']:.3f} / {bb['lg2']:.3f} "
      f"/ {bb['lg3']:.4f}")

The fitted curve rises to a gentle peak around a log income of 8 (roughly \$3,000 per capita), declines through the middle-income range, and flattens — with a barely perceptible uptick — at the top. The scatter is wide: development explains the *shape* but leaves plenty of country-specific variation, which is what the determinants in the next section try to name.

## 9. Turning points and the discriminant test

The cubic in Section 8 *can* bend twice — but does it actually, and does it bend inside the range of incomes we observe? This section answers both. It is the most transferable skill in the post: any time you fit a cubic, these two checks tell you whether the curve really has the shape its coefficients seem to promise. The same two-step test is developed on a synthetic panel in the R companion post [r_kuznets](https://carlos-mendez.org/post/r_kuznets/).

### 9.1 Calculating the turning points

At a turning point the slope is zero, so we set the derivative of the cubic to zero:

$$\frac{\partial \text{GINIW}}{\partial \ln Y} = \beta_1 + 2\beta_2 \ln Y + 3\beta_3 (\ln Y)^2 = 0.$$

This is a quadratic in $\ln Y$, so it has at most two roots — the inverted-U peak and the high-income trough. We solve it and exponentiate each root back into dollars.

In [ ]:
b1, b2, bcub = b3["lg"], b3["lg2"], b3["lg3"]      # 0.293 / -0.032 / 0.00112
D = b2**2 - 3*b1*bcub                                # the discriminant (see 9.2)
roots = np.sort([(-b2 - np.sqrt(D)) / (3*bcub),
                 (-b2 + np.sqrt(D)) / (3*bcub)])      # turning points, in ln Y
print("turning points: ln =", roots.round(2), "->  $", np.exp(roots).round(0))

xg = np.linspace(agg3.lg.min(), agg3.lg.max(), 200)
deriv = b1 + 2*b2*xg + 3*bcub*xg**2
fig, ax = plt.subplots(figsize=(6.6, 4.4))
ax.axhline(0, color=GREY, lw=0.9)
ax.plot(xg, deriv, color=STEEL, lw=2.2)
for r, u in zip(roots, np.exp(roots)):
    ax.axvline(r, color=ORANGE, ls="--", lw=1.4)
    ax.plot([r], [0], "o", color=TEAL, ms=8, zorder=5)
    ax.annotate(f"ln={r:.1f}\n(${u:,.0f})", xy=(r, 0),
                xytext=(r, deriv.max()*0.55), ha="center", color=TEAL,
                fontsize=9, fontweight="bold")
ax.set(xlabel="log GDP per capita", ylabel="marginal effect on inequality",
       title="Where does the regional Kuznets curve turn?")
plt.show()

Regional inequality **rises** with development up to ln(GDP) ≈ 7.7 (about \$2,287), **falls** through the middle-income range until ln(GDP) ≈ 11.3 (about \$77,206), and then **rises again** — both turning points inside the observed range (\$190–\$117,191), the same thresholds the companion [python_fe_kuznets](https://carlos-mendez.org/post/python_fe_kuznets/) post reports. The figure plots the *marginal effect* (the derivative); the turning points are where it crosses zero.

### 9.2 The discriminant: does the curve really bend?

Solving for the roots hides *why* a cubic sometimes has two turning points and sometimes none. The quadratic $\beta_1 + 2\beta_2 Y + 3\beta_3 Y^2 = 0$ has two real solutions exactly when its discriminant is positive:

$$D \equiv \beta_2^2 - 3\beta_1\beta_3.$$

| Discriminant | Real turning points | Shape | Verdict |
|---|---|---|---|
| $D > 0$ | 2 | rise–fall–rise | the cubic shape is **real** |
| $D = 0$ | 1 (inflection) | a single flat spot | knife-edge boundary |
| $D < 0$ | 0 | monotonic | the cubic shape is **not real** |

(The textbook quadratic discriminant is $b^2 - 4ac = 4D$, so the factor of 4 never changes the sign.)

In [ ]:
print(f"D = {D:+.6f}  ->  {'two turning points' if D > 0 else 'monotonic'}")

# Hold the linear and cubic terms fixed; vary ONLY the squared term to flip the regime.
b2_zero = -np.sqrt(3*b1*bcub)                        # the D = 0 knife-edge
regimes = [("D < 0  (monotonic)", -0.025),
           ("D = 0  (inflection)", b2_zero),
           ("D > 0  (genuine N-shape)", b2)]
fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
for ax, (lab, p2) in zip(axes, regimes):
    f = b1*xg + p2*xg**2 + bcub*xg**3
    f = f - f.mean()
    Dp = p2**2 - 3*b1*bcub
    ax.plot(xg, f, color=ORANGE, lw=2.0)
    ax.axhline(0, color=GREY, lw=0.7)
    ax.set_title(f"{lab}\nD = {Dp:+.5f}", fontsize=10)
    ax.set_xlabel("log GDP per capita")
axes[0].set_ylabel("partial fit, centred")
fig.suptitle("Same significant terms, three different shapes", fontweight="bold")
fig.tight_layout()
plt.show()

**The N-shape is real — but only just.** $D = +0.000035$ is barely positive; our cubic sits a hair above the $D=0$ knife-edge, so the high-income upturn is real but faint. A slightly smaller squared term would erase it.

### 9.3 Two checks, not one: significance is not shape

All three income terms are individually significant, but significance answers *"does the data prefer keeping this term?"* — not *"does the curve actually bend inside the range we observe?"* You need the discriminant **and** a check on where the turning points fall. Applying both to our cubic and to three synthetic cases:

In [ ]:
def diagnose(label, b1, b2, b3, lo, hi):
    D = b2**2 - 3*b1*b3
    if D <= 0:
        return dict(case=label, D=D, regime="monotonic (D<0)", in_range=False)
    tp = np.exp(np.sort([(-b2 - np.sqrt(D))/(3*b3), (-b2 + np.sqrt(D))/(3*b3)]))
    ok = bool((tp >= lo).all() and (tp <= hi).all())
    regime = "2 turning points " + ("(both in range)" if ok else "(>=1 OUT of range)")
    return dict(case=label, D=D, regime=regime, in_range=ok)

lo, hi = agg3.GDP_pc_Country.min(), agg3.GDP_pc_Country.max()
rows = [diagnose("This post's cubic (panel FE)",   b1,    b2,    bcub,   lo, hi),
        diagnose("Synthetic A: genuine N-shape",    0.220, -0.026, 0.0010, lo, hi),
        diagnose("Synthetic B: monotonic trap",     0.220, -0.020, 0.0010, lo, hi),
        diagnose("Synthetic C: turns out of range", 0.220, -0.026, 0.0001, lo, hi)]
print(pd.DataFrame(rows).to_string(index=False))

**Significance is not shape.** Reporting "all three terms are significant, so the curve is cubic" can fail two ways — the discriminant can be negative (Synthetic B, monotonic despite the same sign pattern), or the turning points can land outside the data (Synthetic C). The honest workflow: report the coefficients, compute $D$, and *if* $D>0$ confirm the turning points lie inside the observed income range before claiming an inverted-U or N-shape. The R companion [r_kuznets](https://carlos-mendez.org/post/r_kuznets/#7-turning-points-and-the-discriminant-test) extends the same test to a Bayesian model-averaging setting (a high posterior inclusion probability is no more proof of a bend than a significant coefficient is).

## 10. What drives regional inequality?

If two equally rich countries differ in regional inequality, what accounts for the gap? We add blocks of structural controls on top of the cubic — natural resources and farmland, trade and investment openness, geography and transport, aid and schooling, and ethnic inequality — each as its own PyFixest regression with country and period fixed effects. A positive sign means the factor is associated with *more* regional inequality.

In [ ]:
agg4 = collapse5(t4, ["GINIW_pred_GDP_pc", "GDP_pc_Country", "Pop_Country",
                      "Resources_rents_share_of_GDP", "Arable_land",
                      "Trade_GDP_share", "FDI_share_of_GDP", "area",
                      "price_gasoline", "Aid", "School_enrollment_secondary",
                      "GINIW_Eth_light", "Polity2", "fedelupd2"])
agg4["lg"]  = np.log(agg4["GDP_pc_Country"])
agg4["lg2"] = agg4["lg"] ** 2
agg4["lg3"] = agg4["lg"] ** 3
agg4["aid_GDP"] = agg4["Aid"] / (agg4["GDP_pc_Country"] * agg4["Pop_Country"])
agg4["Resources_rents_share_of_GDP"] /= 100
agg4["School_enrollment_secondary"] /= 100
agg4["Area_X_price_gasoline"] = agg4["area"] * agg4["price_gasoline"] / 1e7
agg4["lgXfed"] = agg4["lg"] * agg4["fedelupd2"]          # log GDP x Federal interaction
CUBIC = "lg + lg2 + lg3"


def det_fit(extra):
    return pf.feols(f"GINIW_pred_GDP_pc ~ {CUBIC} + {extra} | Country_ISO+p5",
                    data=agg4, vcov={"CRV1": "Country_ISO"})


d0 = pf.feols(f"GINIW_pred_GDP_pc ~ {CUBIC} | Country_ISO+p5",
              data=agg4, vcov={"CRV1": "Country_ISO"})          # (0) baseline
d1 = det_fit("Resources_rents_share_of_GDP + Arable_land")      # (1) resources
d2 = det_fit("Trade_GDP_share + FDI_share_of_GDP")              # (2) openness
d3 = det_fit("price_gasoline + Area_X_price_gasoline")          # (3) mobility
d_inst = det_fit("Polity2 + lgXfed")                            # (4) institutions
d4 = det_fit("aid_GDP + School_enrollment_secondary")          # (5) transfers + education
d5 = det_fit("GINIW_Eth_light")                                # (6) ethnicity
print("ethnic inequality:", round(d5.coef()["GINIW_Eth_light"], 3), f"(N={int(d5._N)})")

# Table 4 as a side-by-side maketables regression table (baseline + 6 blocks)
T4_LAB = {"GINIW_pred_GDP_pc": "Population-weighted regional Gini",
          "lg": "log GDP p.c.", "lg2": "(log GDP p.c.)^2", "lg3": "(log GDP p.c.)^3",
          "Resources_rents_share_of_GDP": "Resource rents/GDP", "Arable_land": "Arable land",
          "Trade_GDP_share": "Trade/GDP", "FDI_share_of_GDP": "FDI/GDP",
          "price_gasoline": "Gasoline price", "Area_X_price_gasoline": "Area x gasoline",
          "Polity2": "Polity2", "lgXfed": "log GDP x Federal",
          "aid_GDP": "Aid/GDP", "School_enrollment_secondary": "Schooling",
          "GINIW_Eth_light": "Ethnic inequality"}
et4 = mt.ETable([d0, d1, d2, d3, d_inst, d4, d5],
                model_heads=["baseline", "resources", "openness", "mobility",
                             "institutions", "transfers/edu", "ethnicity"],
                labels=T4_LAB, felabels={"Country_ISO": "Country FE", "p5": "Period FE"},
                coef_fmt="b:.3f* (se:.3f)", show_fe=True)
et4.make("gt")

The strongest correlate by far is **ethnic inequality** at **0.071** (p < 0.001): countries where income differs sharply across ethnic homelands also have sharply unequal regions. Resource rents push inequality up (0.018, p < 0.01) — resource wealth concentrates in a few regions — while a larger arable-land share pulls it down (−0.053, p < 0.001), consistent with agriculture spreading income more evenly. One column of the paper (institutional quality, from the licensed ICRG database) cannot be reproduced and is omitted. The sample size drifts across columns as different controls go missing, so the columns should be read as separate windows, not a single nested model.

## 11. Spatial robustness: Conley standard errors

Regions are not independent: a boom in one province spills into its neighbours, so their regression errors are correlated. Ignoring that makes standard errors too small and t-statistics too big. We re-estimate the clean light elasticity (column 2, β = 0.190) and recompute its standard error allowing errors of regions within a chosen radius to be correlated — the **Conley** spatial-HAC correction — using a from-scratch implementation based on great-circle distances between region centroids.

In [ ]:
tb4["satyear"] = sum(i * tb4[f"satyear_{i}"] for i in range(1, 8)).astype(int)
cols = ["log_GDP_pc_Region", "log_Light_ppix_Region", "Latitude", "Longitude",
        "code_Coutry_Region", "satyear"]
dfb = tb4.dropna(subset=cols).copy()
cnt = dfb["code_Coutry_Region"].value_counts()
dfb = dfb[dfb["code_Coutry_Region"].isin(cnt[cnt > 1].index)].copy()
mb = pf.feols("log_GDP_pc_Region ~ log_Light_ppix_Region | "
              "code_Coutry_Region + satyear", data=dfb)
b_b4 = float(mb.coef()["log_Light_ppix_Region"])

# Frisch-Waugh residualisation for the single-regressor Conley variance.
xfit = pf.feols("log_Light_ppix_Region ~ 1 | code_Coutry_Region + satyear",
                data=dfb)
xt = np.asarray(xfit.resid())
u  = np.asarray(mb.resid())
xu = xt * u
sxx = float(np.sum(xt ** 2))
gb = dfb.assign(xu=xu).groupby("code_Coutry_Region")
agg_s = gb["xu"].sum().to_numpy()
lat = np.radians(gb["Latitude"].first().to_numpy())
lon = np.radians(gb["Longitude"].first().to_numpy())
R_EARTH, nr2 = 6371.0, agg_s.size


def conley_var(dist_km):
    """Conley spatial-HAC variance: Bartlett-weight cross-products of region
    scores by great-circle (haversine) distance, zero beyond dist_km."""
    v = 0.0
    for i in range(nr2):
        a = (np.sin((lat - lat[i]) / 2) ** 2
             + np.cos(lat[i]) * np.cos(lat) * np.sin((lon - lon[i]) / 2) ** 2)
        dij = 2 * R_EARTH * np.arcsin(np.minimum(1.0, np.sqrt(a)))
        w = np.maximum(0.0, 1 - dij / dist_km)         # Bartlett kernel
        v += agg_s[i] * np.sum(w * agg_s)
    return v / (sxx ** 2)


radii = [1000, 2500, 5000]
se_conley = {r: float(np.sqrt(conley_var(r))) for r in radii}
iid_se = float(mb.se()["log_Light_ppix_Region"])
print(f"point estimate = {b_b4:.3f}")
for r in radii:
    print(f"Conley SE @ {r} km = {se_conley[r]:.3f}")
print(f"naive (iid) SE     = {iid_se:.3f}")

fig, ax = plt.subplots(figsize=(6.6, 4))
labels = ["iid (default)"] + [f"Conley {r} km" for r in radii]
ses = [iid_se] + [se_conley[r] for r in radii]
colors = [GREY, STEEL, STEEL, STEEL]
for i, (s, c) in enumerate(zip(ses, colors)):
    ax.plot([b_b4 - 1.96 * s, b_b4 + 1.96 * s], [i, i], color=c, lw=3,
            solid_capstyle="round")
    ax.plot(b_b4, i, "o", color=INK, ms=6)
ax.axvline(0, color=ORANGE, lw=1.5, ls="--")
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
ax.set(xlabel="light elasticity (95% CI)",
       title=f"Spatial robustness of the light elasticity (β = {b_b4:.3f})")
ax.invert_yaxis()
fig.tight_layout()
plt.show()

Allowing for spatial correlation roughly doubles to triples the standard error — from 0.013 to between 0.026 and 0.037 — because neighbouring regions are not the independent observations the naive formula assumes. Even so, the elasticity of 0.190 stays far from zero (a t-statistic above 5 at the widest radius), so the lights-predict-income relationship is not a statistical mirage created by ignoring geography.

## 12. Regional versus personal inequality

A natural question: is *regional* inequality (gaps between places) just a reflection of *personal* inequality (gaps between people)? We compare each country's regional Gini with its household-income Gini, both averaged over 2001–2012, and fit a line. A positive slope means the two inequalities go together.

In [ ]:
d5f = f5[(f5.year > 2000) & (f5.year < 2013)].copy()
agg5 = d5f.groupby("Country_ISO", as_index=False)[
    ["GINIW_pred_GDP_pc", "Giniall"]].mean()
agg5["GINIall_100"] = agg5["Giniall"] / 100
p5d = agg5.dropna(subset=["GINIall_100", "GINIW_pred_GDP_pc"])
slope, icept = np.polyfit(p5d["GINIW_pred_GDP_pc"], p5d["GINIall_100"], 1)
print(f"n = {len(p5d)} countries | OLS slope = {slope:.3f}")

fig, ax = plt.subplots(figsize=(6, 4.8))
ax.scatter(p5d["GINIW_pred_GDP_pc"], p5d["GINIall_100"], marker="^", s=26,
           facecolors="none", edgecolors=STEEL)
xs = np.linspace(p5d["GINIW_pred_GDP_pc"].min(),
                 p5d["GINIW_pred_GDP_pc"].max(), 200)
ax.plot(xs, slope * xs + icept, color=ORANGE, lw=2.2, label="OLS fit")
ax.set(xlabel="interregional inequality (GINIW)",
       ylabel="interpersonal inequality (household Gini)",
       title="Regional vs personal inequality (Figure 5a)")
ax.legend(loc="upper left", frameon=False)
fig.tight_layout()
plt.show()

Across 144 countries the household-income Gini rises with the regional Gini at a slope of **0.587**: places with wide gaps *between regions* also tend to have wide gaps *between people*. Regional and personal inequality are distinct but linked — so policies that narrow the gap between a country's regions are also, in part, distributional policies between its citizens.

## 13. Discussion

We set out to answer a measurement question — can we see inside countries from space? — and a substantive one — how does regional inequality move with development? The answer to the first is a qualified yes: a light-to-income elasticity of 0.102, predictions that correlate 0.925 with observed income, and inequality measures that track the observed data twice as well as raw light does. The method turns a data desert into a global, comparable income map.

On the substantive question, regional inequality follows an N-shaped Kuznets path — rising through early development, falling as countries converge internally (the world average dropped from 0.070 to 0.061 over 1992–2012), with a faint upturn among the very richest. The single strongest correlate is ethnic inequality (0.071), a reminder that the internal economic geography of a country is bound up with its human geography. Two cautions: the relationships are descriptive associations with fixed effects, not causal effects; and the income figures are *predictions*, accurate on average but wrong for any single unusual region.

## 14. Summary and next steps

- **Method insight.** Nighttime lights predict regional income with an elasticity of 0.102 and a 0.925 correlation with observed income; predicting income first, rather than equating light with income, doubles the quality of the resulting inequality measures (Gini correlation 0.49 vs 0.21).
- **Measurement insight.** Population weighting is not cosmetic: weighted and unweighted Gini correlate only 0.75, and weighting lowers measured inequality by ~0.003 on average, so the weighting choice must be reported.
- **Substantive insight.** The regional Kuznets curve is N-shaped (cubic 0.293 / −0.032 / 0.001 across 180 countries), and ethnic inequality (0.071) is its strongest structural correlate.
- **Robustness insight.** The light elasticity of 0.190 survives spatial correlation — Conley standard errors of 0.026–0.037 are two to three times the naive 0.013, but the estimate stays far from zero.
- **Limitation.** Our from-scratch indices use only the ~1,500 regions with observed income; the published series uses every region on Earth (correlation 0.88).
- **Next steps.** Swap in a modern lights product (VIIRS replacing DMSP) to extend the series past 2012; or carry the full-world prediction through to rebuild the global income map.

## 15. Exercises

1. **Re-weight the world.** Modify `ineq_indices` to weight regions by land area instead of population, recompute the regional Gini for every country, and compare the cross-country ranking to the population-weighted one. Which countries move most, and why?
2. **A fourth act?** Re-estimate the Kuznets cubic on the coefficient of variation (`COVW_pred_GDP_pc`) instead of the Gini, and add a quartic term (`lg4`). Does the upturn at high income strengthen, vanish, or stay a rounding error?
3. **How far do shocks travel?** Recompute the Conley standard error at radii of 250, 500, and 10,000 km. Plot the standard error against the radius. At what distance does spatial correlation stop mattering for the light elasticity?

## 16. References

1. [Lessmann, C., & Seidel, A. (2017). Regional inequality, convergence, and its determinants — A view from outer space. *European Economic Review*, 92, 110–132.](https://doi.org/10.1016/j.euroecorev.2016.11.009)
2. [Henderson, J. V., Storeygard, A., & Weil, D. N. (2012). Measuring economic growth from outer space. *American Economic Review*, 102(2), 994–1028.](https://doi.org/10.1257/aer.102.2.994)
3. [Gennaioli, N., La Porta, R., Lopez-de-Silanes, F., & Shleifer, A. (2014). Growth in regions. *Journal of Economic Growth*, 19(3), 259–309.](https://doi.org/10.1007/s10887-014-9105-9)
4. [Kuznets, S. (1955). Economic growth and income inequality. *American Economic Review*, 45(1), 1–28.](https://www.jstor.org/stable/1811581)
5. [Conley, T. G. (1999). GMM estimation with cross sectional dependence. *Journal of Econometrics*, 92(1), 1–45.](https://doi.org/10.1016/S0304-4076(98)00084-0)
6. [PyFixest — fast fixed-effects estimation in Python (documentation)](https://py-econometrics.github.io/pyfixest/)
7. [linearmodels — panel data models in Python (documentation)](https://bashtage.github.io/linearmodels/)

## Appendix A. Data dictionary

This appendix documents **every column in all six data files**: what it is, how it was originally
constructed, its source, its units, and its **time–country coverage**. Coverage is written as
*years · units · N*, where *N* is the number of non-missing observations and *units* counts the
distinct countries (country files) or regions (region files) with data. Definitions follow Lessmann
and Seidel (2017) and the official variable labels in the authors' replication archive; coverage and
statistics are computed in `script.py`.

### A.1 The six datasets in detail

All six files are tidy panels. The **region files** are keyed by region × year over the
1,504-region / 81-country training frame (1992–2010); the **country files** are keyed by
`Country_ISO` × year over 180 countries (1992–2012). The inequality indices in the country files are
built from the predicted regional incomes in the region files.

| File | Unit | Rows × Cols | Years | Countries | Regions | What it is for |
|---|---|---|---|---|---|---|
| `Prediction_Data.csv` | region-year | 5,258 × 30 | 1992–2010 | 81 | 1,504 | Train the light→income model (Table 1) |
| `Table_2_data.csv` | region-year | 5,258 × 8 | 1992–2010 | 81 | 1,504\* | Validate the inequality indices (Table 2) |
| `Table_3_data.csv` | country-year | 3,675 × 9 | 1992–2012 | 180 | — | Kuznets curve: GDP + 5 indices (Table 3) |
| `Table_4_data.csv` | country-year | 3,675 × 17 | 1992–2012 | 180 | — | Determinants of inequality (Table 4) |
| `Table_B4_data.csv` | region-year | 5,258 × 14 | 1992–2010 | 81 | 1,504 | Conley spatial-HAC errors (+ lat/lon) |
| `Figure_5_data.csv` | country-year | 3,675 × 5 | 1992–2012 | 180 | — | Regional vs personal inequality |

\* `Table_2_data.csv` has no explicit region-id column, but its rows are the same 1,504-region
training frame at region-year.

### A.2 Variable dictionary

Coverage shorthand: region-frame variables are **1992–2010 · 1,504 reg (81 ctry) · N = 5,258** unless
noted; core country-frame variables are **1992–2012 · 180 ctry · N = 3,675** unless noted.

**Identifiers and keys**

| Variable | What it is | How constructed | Source | Unit | Coverage |
|---|---|---|---|---|---|
| `Country_ISO` | Country code (ISO 3166-1) | Assigned per country | GADM | string | all files |
| `Country_NAME` | Country name | — | GADM | string | all files |
| `Region_NAME` | Region name | 1st-level admin-unit name | GADM | string | region frame |
| `code_Coutry_Region` | Numeric region key (original spelling kept) | Region identifier | Authors | integer | region frame |
| `id_t_j` | Country-year key | Concatenation of year + ISO (e.g. `2010CHE`) | Authors | string | region frame |
| `year` | Calendar year | — | — | year | per file (see A.1) |

**Lights and income**

| Variable | What it is | How constructed | Source | Unit | Coverage |
|---|---|---|---|---|---|
| `log_Light_ppix_Region` | Log avg nighttime light per pixel | Region mean of DMSP-OLS stable-lights DN (0–63); +0.01 if zero, then log | NOAA/NGDC | log DN | region frame |
| `Light_Region` | Regional total lights | Sum of pixel DN over the region | NOAA/NGDC | summed DN | 1992–2010 · 81 ctry · 5,258 (Table_2) |
| `Light_Country` | Country total lights | Sum of pixel DN over the country | NOAA/NGDC | summed DN | 1992–2010 · 81 ctry · 5,258 (Table_2) |
| `GDP_pc_Region` | Observed regional GDP per capita | Regional accounts, constant 2005 PPP US\$ | Gennaioli et al. (2014) | US\$ | region frame |
| `log_GDP_pc_Region` | Log of `GDP_pc_Region` | Natural log | Gennaioli et al. (2014) | log US\$ | region frame |
| `pred_GDP_pc_Region` | Predicted regional GDP per capita | Fitted values of the eq.-1 RE model applied to all regions | This paper (model) | US\$ | 1992–2010 · 81 ctry · 5,258 (Table_2) |
| `GDP_pc_Country` | National GDP per capita | constant 2005 PPP US\$ | World Bank WDI | US\$ | country frame |
| `log_GDP_pc_Country` | Log national GDP per capita | Natural log | World Bank WDI | log US\$ | region frame |

**Prediction-model regressors and fixed-effect dummies**

| Variable | What it is | How constructed | Source | Unit | Coverage |
|---|---|---|---|---|---|
| `log_N_pix_top_cod_1_ppix` | Log # top-coded pixels (DN = 63) | Count of saturated pixels per region, logged | NOAA/NGDC | log count | region frame |
| `log_N_pix_low_cod_1_ppix` | Log # low-coded pixels (DN = 0) | Count of dark pixels per region, logged | NOAA/NGDC | log count | region frame |
| `log_area` | Log region area | Region polygon area, logged | GADM | log km² | region frame |
| `log_region` | Log # regions in the country | Count of regions per country, logged | GADM / Gennaioli | log count | region frame |
| `log_region_X_log_area` | Interaction term | `log_region` × `log_area` | Derived | — | region frame |
| `eap`, `eca`, `lac`, `mena`, `sa`, `ssa` | World-Bank region-group dummies | 1 if the country is in that group (North America = reference) | World Bank | 0/1 | region frame |
| `satyear_1` … `satyear_7` | Satellite-configuration dummies | 1 per satellite/sensor era (sensors change and age over time) | NOAA/NGDC | 0/1 | region frame |

**Population and geography**

| Variable | What it is | How constructed | Source | Unit | Coverage |
|---|---|---|---|---|---|
| `Pop_Region` | Regional total population | Population density × region area, rounded up (min 1); 5-yr waves interpolated to annual | GPW v3 (CIESIN) | persons | region frame |
| `Pop_Country` | Country total population | Sum of regional populations | GPW v3 (CIESIN) | persons | region & country frames |
| `area` | Country land area | Total land area (excl. inland water) | World Bank WDI | km² | country frame |
| `Latitude`, `Longitude` | Region centroid coordinates | Polygon centroid | GADM | degrees | region frame (Table_B4) |

**Inequality indices** (all population-weighted, on predicted regional income)

| Variable | What it is | How constructed | Source | Unit | Coverage |
|---|---|---|---|---|---|
| `GINIW_pred_GDP_pc` | Regional Gini | Population-weighted Gini of `pred_GDP_pc_Region` within a country-year | This paper | 0–1 | country frame |
| `COVW_pred_GDP_pc` | Regional coefficient of variation | Population-weighted CV | This paper | ≥ 0 | country frame |
| `GE_1W_pred_GDP_pc` | Theil index, GE(1) | Population-weighted GE(α = 1) | This paper | ≥ 0 | country frame |
| `GE_0W_pred_GDP_pc` | Mean log deviation, GE(0) | Population-weighted GE(α = 0) | This paper | ≥ 0 | country frame |
| `GE_m1W_pred_GDP_pc` | GE(−1) | Population-weighted GE(α = −1) | This paper | ≥ 0 | country frame |
| `Giniall` | National interpersonal income Gini | Household-survey income Gini (0–100 scale) | Lessmann & Seidel (2017) | 0–100 | 1992–2012 · 153 ctry · 1,330 (Figure_5) |

**Determinants** (country frame, 1992–2012)

| Variable | What it is | How constructed | Source | Unit | Coverage |
|---|---|---|---|---|---|
| `Resources_rents_share_of_GDP` | Natural-resource rents | Oil + gas + coal + mineral + forest rents, % of GDP | World Bank WDI | % GDP | 177 ctry · N = 3,620 |
| `Arable_land` | Arable-land share | Arable land as a share of land area (FAO definition) | World Bank WDI | share | 178 ctry · N = 3,603 |
| `Trade_GDP_share` | Trade openness | (Exports + imports) / GDP | World Bank WDI | ratio | 176 ctry · N = 3,509 |
| `FDI_share_of_GDP` | FDI openness | Net FDI inflows / GDP | World Bank WDI | ratio | 174 ctry · N = 3,477 |
| `price_gasoline` | Gasoline pump price | Pump price, PPP constant 2005 US\$/litre (the paper's "transport cost" = area × price) | World Bank WDI | US\$/L | 162 ctry · N = 1,366 |
| `Aid` | Aid flows | Net aid received, constant 2011 US\$ | World Bank WDI | US\$ | 155 ctry · N = 2,964 |
| `School_enrollment_secondary` | Secondary-school enrolment | Gross secondary enrolment ratio | World Bank WDI | % gross | 172 ctry · N = 2,566 |
| `GINIW_Eth_light` | Ethnic inequality | Population-weighted light-Gini across ethnic homelands (method of Alesina et al. 2016) | NOAA/NGDC + GREG (Weidmann et al. 2010) | 0–1 | 173 ctry · N = 3,528 |
| `Polity2` | Democracy–autocracy score | Polity IV combined score, rescaled −1 (autocracy) to +1 (democracy) | Center for Systemic Peace, Polity IV | −1…+1 | 157 ctry · N = 3,158 |
| `fedelupd2` | Federalism dummy | 1 if the country is federally organised | Authors | 0/1 | 1992–2009 · 154 ctry · N = 2,724 |

The authors' Table 4 also uses an ICRG "bureaucratic quality" index, which is licensed and **not**
redistributed in this bundle; the post therefore omits that one determinant column.